# Data Transformation Assignment
**Dataset:** `employee_productivity_dataset.csv`

**Objective:** Prepare the given dataset for Machine Learning by applying Encoding, Normalization, Standardization (Scaling), and a Preprocessing Pipeline.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MinMaxScaler, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

df = pd.read_csv('employee_productivity_dataset.csv')
print("Dataset shape:", df.shape)
display(df.head())

## Q1. Handle Missing Values (Basic Cleaning)
**Question:** Fill missing values using:
- Age → Median
- Salary → Mean
- Hours_Worked_Per_Week → Median
- Performance_Score → Mean

Display the dataset after handling missing values.

### Solution
The specified imputation methods are applied directly to the dataset.

In [ ]:
print("Missing values before:")
display(df[['Age','Salary','Hours_Worked_Per_Week','Performance_Score']].isnull().sum())

df['Age'] = df['Age'].fillna(df['Age'].median())
df['Salary'] = df['Salary'].fillna(df['Salary'].mean())
df['Hours_Worked_Per_Week'] = df['Hours_Worked_Per_Week'].fillna(df['Hours_Worked_Per_Week'].median())
df['Performance_Score'] = df['Performance_Score'].fillna(df['Performance_Score'].mean())

print("Missing values after:")
display(df[['Age','Salary','Hours_Worked_Per_Week','Performance_Score']].isnull().sum())
display(df)

**Result from the supplied dataset:** Missing values before cleaning were `{'Age': 25, 'Salary': 25, 'Hours_Worked_Per_Week': 50, 'Performance_Score': 50}`. After applying the required methods, all four columns have 0 missing values: `{'Age': 0, 'Salary': 0, 'Hours_Worked_Per_Week': 0, 'Performance_Score': 0}`.

## Q2. Label Encoding
**Question:** Convert Gender and Department into numeric values using Label Encoding. Show the updated columns.

### Solution
Label Encoding assigns a unique integer to every category.

In [ ]:
df_label = df.copy()
for column in ['Gender','Department']:
    le = LabelEncoder()
    df_label[column] = le.fit_transform(df_label[column].astype(str))
    print(column, dict(zip(le.classes_, le.transform(le.classes_))))
display(df_label[['Gender','Department']].head(10))

**Mappings obtained from the supplied dataset:** Gender = `{'Female': 0, 'Male': 1}`; Department = `{'Finance': 0, 'HR': 1, 'IT': 2, 'Marketing': 3}`.

## Q3. One-Hot Encoding
**Question:** Apply One-Hot Encoding on Work_Mode and Location. Display the dataset and check how many new columns are created.

### Solution
One-Hot Encoding converts each category into a separate binary indicator column.

In [ ]:
original_columns = df.shape[1]
df_onehot = pd.get_dummies(df, columns=['Work_Mode','Location'], dtype=int)
dummy_columns = [c for c in df_onehot.columns if c.startswith('Work_Mode_') or c.startswith('Location_')]
print("Dummy columns created:", len(dummy_columns))
print(dummy_columns)
print("Final number of columns:", df_onehot.shape[1])
display(df_onehot.head())

**Result:** 7 dummy columns are produced for Work_Mode and Location: `['Work_Mode_Hybrid', 'Work_Mode_Onsite', 'Work_Mode_Remote', 'Location_Bangalore', 'Location_Delhi', 'Location_Mumbai', 'Location_Pune']`. The encoded dataset has 17 columns.

## Q4. Normalization (Min-Max Scaling)
**Question:** Normalize Salary and Hours_Worked_Per_Week using MinMaxScaler and display the results.

### Solution
Min-Max scaling converts values to the range 0 to 1.

In [ ]:
df_minmax = df.copy()
scaler = MinMaxScaler()
df_minmax[['Salary','Hours_Worked_Per_Week']] = scaler.fit_transform(
    df_minmax[['Salary','Hours_Worked_Per_Week']]
)
display(df_minmax[['Salary','Hours_Worked_Per_Week']].head(10))
print("Minimums:")
print(df_minmax[['Salary','Hours_Worked_Per_Week']].min())
print("Maximums:")
print(df_minmax[['Salary','Hours_Worked_Per_Week']].max())

## Q5. Standardization (Scaling)
**Question:** Apply StandardScaler on Age and Projects_Completed. Display the transformed values.

### Solution
StandardScaler transforms each feature so that it has approximately mean 0 and standard deviation 1.

In [ ]:
df_standard = df.copy()
scaler = StandardScaler()
df_standard[['Age','Projects_Completed']] = scaler.fit_transform(
    df_standard[['Age','Projects_Completed']]
)
display(df_standard[['Age','Projects_Completed']].head(10))
print("Means:", df_standard[['Age','Projects_Completed']].mean().to_dict())
print("Standard deviations:", df_standard[['Age','Projects_Completed']].std(ddof=0).to_dict())

## Q6. Compare Scaling Methods
**Question:** Apply both MinMaxScaler and StandardScaler on Salary. Show both results side by side.

### Solution
MinMaxScaler maps Salary into 0–1, whereas StandardScaler expresses Salary relative to its mean and standard deviation.

In [ ]:
comparison = pd.DataFrame({'Original_Salary': df['Salary']})
comparison['MinMax_Salary'] = MinMaxScaler().fit_transform(df[['Salary']]).ravel()
comparison['Standardized_Salary'] = StandardScaler().fit_transform(df[['Salary']]).ravel()
display(comparison.head(15))

## Q7. Build Preprocessing Pipeline
**Question:** Create a pipeline that applies encoding to categorical columns and scaling to numerical columns using ColumnTransformer and Pipeline.

### Solution
The pipeline automatically preprocesses numeric and categorical features in one reusable workflow.

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
categorical_columns = df.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_columns),
    ('cat', categorical_pipeline, categorical_columns)
])

pipeline = Pipeline([
    ('preprocessor', preprocessor)
])

print("Numerical columns:", numeric_columns)
print("Categorical columns:", categorical_columns)
print(pipeline)

## Q8. Apply Pipeline
**Question:** Apply the pipeline on the dataset. Display the transformed dataset and shape of final dataset.

### Solution

In [ ]:
transformed_data = pipeline.fit_transform(df)
if hasattr(transformed_data, 'toarray'):
    transformed_data = transformed_data.toarray()

feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()
transformed_df = pd.DataFrame(transformed_data, columns=feature_names)

display(transformed_df.head(10))
print("Original shape:", df.shape)
print("Final transformed shape:", transformed_df.shape)

**Result from the supplied dataset:** Original shape = **(250, 12)** and final transformed shape = **(250, 48)**.

## Q9. Conceptual Question
**Question:** Why is scaling important in Python?

### Solution
Scaling places numerical features on comparable scales. This prevents variables with large numerical ranges, such as Salary, from dominating variables with smaller ranges. It is especially important for distance-based and gradient-based machine-learning algorithms and can improve numerical stability and model convergence.

## Q10. Conceptual Question
**Question:** Why do we convert categorical data into numerical form?

### Solution
Most machine-learning algorithms work with numerical values rather than text categories. Encoding converts categories such as Gender, Department, Work_Mode, and Location into numerical representations that a model can process. Label Encoding is useful for suitable categorical representations, while One-Hot Encoding avoids creating an artificial order among nominal categories.

# Conclusion
The supplied Employee Productivity dataset was prepared for machine learning by handling missing values, applying Label Encoding and One-Hot Encoding, performing Min-Max normalization and standardization, comparing scaling methods, and building a complete preprocessing pipeline.